# Deploy Fine-tuned VLM to Vertex AI Endpoint

End-to-end deployment of a fine-tuned Qwen2.5-VL LoRA adapter to a live Vertex AI endpoint.

**Flow:**
1. Config — all variables in one place, edit here only
2. Init GCP + select model checkpoint from GCS
3. Build & push Docker image (via GCP Cloud Shell / Cloud Build — Windows has no Docker daemon)
4. Deploy model to Vertex AI Endpoint
5. (Optional) Debug helpers — quota check, GPU availability, image inspection

**Architecture:** container runs `vertex-entrypoint.sh` → downloads LoRA adapter from GCS → merges with base model (`swift export --merge_lora`) → serves via `vllm serve` on port 8080, OpenAI-compatible (`/v1/chat/completions`, `/health`).

## 0. Config

**Edit only this cell** to change project, region, model checkpoint, GPU type, or endpoint name. Every other cell reads from these variables — nothing else should be hardcoded below.

In [33]:
import json

# loaded from gcs_config.json, same file used in notebooks 02-05
with open("gcs_config.json") as f:
    gcs_cfg = json.load(f)

PROJECT_ID          = gcs_cfg["project_id"]
REGION               = "asia-southeast1"              # ← changed from asia-southeast2
BUCKET_NAME          = gcs_cfg["bucket_name"]
GCS_ROOT_URI         = f"gs://{BUCKET_NAME}"
GCS_MODEL_DIR        = f"{GCS_ROOT_URI}/output/model"   # where train.py / Vertex training job writes checkpoints

# Docker image (built on Cloud Shell / Cloud Build, see Section 2)
REPO_NAME            = "swift-json-vlm-container-finetuned"
IMAGE_URI            = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO_NAME}/inference:latest"

# Vertex AI endpoint / model registry names
ENDPOINT_NAME        = "document-to-json"
DEPLOYED_MODEL_NAME  = "document-to-json"

# Serving compute. T4 16GB is sufficient for Qwen2.5-VL-3B merged model via vLLM
MACHINE_TYPE         = "n1-standard-8"
ACCELERATOR_TYPE     = "NVIDIA_TESLA_T4"
ACCELERATOR_COUNT    = 1

# Deploy retry behaviour (Vertex AI GPU capacity can be transiently unavailable)
MAX_RETRIES          = 5
RETRY_WAIT_SEC        = 60
DEPLOY_TIMEOUT_SEC    = 1800

# Vertex AI prediction service account (fixed per project, update if SA changes)
VERTEX_SA        = "custom-online-prediction@a00ba840168c92a5f-tp.iam.gserviceaccount.com"

# Cloud Build service account (project_number-compute@developer.gserviceaccount.com)
CLOUDBUILD_SA    = f"73397202200-compute@developer.gserviceaccount.com"

# Pre-merged model URI — model already merged, no swift export needed at serve time
MERGED_MODEL_URI = f"{GCS_ROOT_URI}/output/merged/v0-20260623-161620"  # ← use merged model directly

# vLLM / serving container env vars (read by vertex-entrypoint.sh, SM_VLLM_* -> vllm serve args)
SERVING_ENV_EXTRA = {
    "USE_HF_TRANSFER":             "true",
    "HF_HUB_ENABLE_HF_TRANSFER":   "1",
    "SIZE_FACTOR":                 "8",
    "MAX_PIXELS":                  "602112",
    "SM_VLLM_SERVED_MODEL_NAME":   DEPLOYED_MODEL_NAME,
    "SM_VLLM_LIMIT_MM_PER_PROMPT": "image=2, video=0",
    "SM_VLLM_MAX_MODEL_LEN":       "8192",
    "SM_VLLM_MAX_NUM_SEQS":        "2",
    "SM_VLLM_DTYPE":               "float16",  # required on T4 (no native bfloat16 support)
}

print(f"Project        : {PROJECT_ID}")
print(f"Region         : {REGION}")
print(f"GCS model dir  : {GCS_MODEL_DIR}")
print(f"Merged model   : {MERGED_MODEL_URI}")
print(f"Image URI      : {IMAGE_URI}")
print(f"Endpoint name  : {ENDPOINT_NAME}")
print(f"Serving compute: {MACHINE_TYPE} + {ACCELERATOR_TYPE} x{ACCELERATOR_COUNT}")


Project        : first-orc-chien
Region         : asia-southeast1
GCS model dir  : gs://electric-bill-dataset-gcs/output/model
Merged model   : gs://electric-bill-dataset-gcs/output/merged/v0-20260623-161620
Image URI      : asia-southeast1-docker.pkg.dev/first-orc-chien/swift-json-vlm-container-finetuned/inference:latest
Endpoint name  : document-to-json
Serving compute: n1-standard-8 + NVIDIA_TESLA_T4 x1


## 1. Setup

Install SDKs and initialize the Vertex AI + GCS clients using the config above.

In [34]:
# One-time install — uncomment if not already installed in this environment
# !pip install google-cloud-aiplatform google-cloud-storage huggingface_hub --quiet

In [35]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [36]:
import os
from google.cloud import aiplatform, storage
from utils.model_manager import list_gcs_models

aiplatform.init(project=PROJECT_ID, location=REGION)
gcs_client = storage.Client(project=PROJECT_ID)

print("✅ Vertex AI + GCS clients initialized")

✅ Vertex AI + GCS clients initialized


## 2. Select Model Checkpoint

List fine-tuned checkpoints under `GCS_MODEL_DIR` and pick which one to deploy.

In [37]:
df_models = list_gcs_models(GCS_MODEL_DIR, project_id=PROJECT_ID)
df_models

,Key
0,gs://electric-bill-dataset-gcs/output/model/v0...


In [38]:
# Index into df_models above — change WHICH_MODEL to select a different checkpoint
WHICH_MODEL = 0
model_output_url = df_models["Key"].iloc[WHICH_MODEL].rstrip("/")
print(f"Selected checkpoint: {model_output_url}")

Selected checkpoint: gs://electric-bill-dataset-gcs/output/model/v0-20260623-161620


## 3. Build & Push Docker Image

Vertex AI serves custom LoRA models via a **custom container** pushed to Artifact Registry. The container bundles:
- ms-swift (merges the LoRA adapter into the base model at startup)
- vLLM (high-throughput GPU inference server)
- `vertex-entrypoint.sh`: downloads the adapter from `ADAPTER_URI` (GCS), runs `swift export --merge_lora`, then starts `vllm serve` on port 8080

**Windows has no Docker daemon**, so the image is built remotely on GCP (Cloud Shell or Cloud Build), not from this notebook.

In [39]:
print(f"""
STEP 1 — Open Cloud Shell
  https://shell.cloud.google.com/

STEP 2 — Upload these 4 files (from docker-artifacts/):
  Dockerfile, vertex-entrypoint.sh, 01_docker_install.sh, all merge-job (merge base model + checkpoint)

STEP 3 — Build & push Docker image:
  cd docker-artifacts
  gcloud config set project {PROJECT_ID}

  gcloud artifacts repositories create {REPO_NAME} \\
      --repository-format=docker --location={REGION} --project={PROJECT_ID}

  gcloud projects add-iam-policy-binding {PROJECT_ID} \\
      --member="serviceAccount:{CLOUDBUILD_SA}" \\
      --role="roles/artifactregistry.writer"

  gcloud projects add-iam-policy-binding {PROJECT_ID} \\
      --member="serviceAccount:{CLOUDBUILD_SA}" \\
      --role="roles/logging.logWriter"

  sed -i 's/\\r//' vertex-entrypoint.sh

 .

STEP 3.5 — Merge LoRA adapter with base model (run once, ~15-20 min):
 
   gcloud builds submit     
      --tag {IMAGE_URI}
      --machine-type=E2_HIGHCPU_8     
      --disk-size=100     
      --timeout=3600       
      --project={PROJECT_ID}

  Merged model will be at: {MERGED_MODEL_URI}

STEP 4 — Grant Vertex AI service account access to GCS bucket:
  gcloud storage buckets add-iam-policy-binding \\
      gs://{BUCKET_NAME} \\
      --member="serviceAccount:{VERTEX_SA}" \\
      --role="roles/storage.objectViewer"

IMAGE URI (already set from config):
  {IMAGE_URI}
""")


STEP 1 — Open Cloud Shell
  https://shell.cloud.google.com/

STEP 2 — Upload these 4 files (from docker-artifacts/):
  Dockerfile, vertex-entrypoint.sh, 01_docker_install.sh, all merge-job (merge base model + checkpoint)

STEP 3 — Build & push Docker image:
  cd docker-artifacts
  gcloud config set project first-orc-chien

  gcloud artifacts repositories create swift-json-vlm-container-finetuned \
      --repository-format=docker --location=asia-southeast1 --project=first-orc-chien

  gcloud projects add-iam-policy-binding first-orc-chien \
      --member="serviceAccount:73397202200-compute@developer.gserviceaccount.com" \
      --role="roles/artifactregistry.writer"

  gcloud projects add-iam-policy-binding first-orc-chien \
      --member="serviceAccount:73397202200-compute@developer.gserviceaccount.com" \
      --role="roles/logging.logWriter"

  sed -i 's/\r//' vertex-entrypoint.sh

 .

STEP 3.5 — Merge LoRA adapter with base model (run once, ~15-20 min):

   gcloud builds submit 

In [40]:
!gcloud artifacts docker images list asia-southeast1-docker.pkg.dev/first-orc-chien/swift-json-vlm-container-finetuned --include-tags

IMAGE                                                                                        DIGEST                                                                   TAGS    CREATE_TIME          UPDATE_TIME          SIZE
asia-southeast1-docker.pkg.dev/first-orc-chien/swift-json-vlm-container-finetuned/inference  sha256:29d3b024c20f92c08353d09ba31812eb50f28ce4564d312f2c2a97634fb4e0dd          2026-06-29T16:30:14  2026-07-01T00:58:37  11596977598
asia-southeast1-docker.pkg.dev/first-orc-chien/swift-json-vlm-container-finetuned/inference  sha256:35bf4ff7aaa7c2fdf6ea6a13b3cd6d755a179521516f2d31871cc70d3fa9ae3a  latest  2026-07-01T14:25:32  2026-07-01T14:25:32  11779224654
asia-southeast1-docker.pkg.dev/first-orc-chien/swift-json-vlm-container-finetuned/inference  sha256:465ae744bef907016a880d89d1a9f2c0b3469c522f81f389f6e2c49e31adba2e          2026-07-01T12:39:41  2026-07-01T14:25:32  11779224015
asia-southeast1-docker.pkg.dev/first-orc-chien/swift-json-vlm-container-finetuned/inference  sh

Listing items under project first-orc-chien, location asia-southeast1, repository swift-json-vlm-container-finetuned.



In [41]:
# ADAPTER_URI is what vertex-entrypoint.sh reads inside the container to pull the model from GCS
# Points to pre-merged model — no LoRA merging needed at serve time
os.environ["REPO_NAME"]   = REPO_NAME
os.environ["ADAPTER_URI"] = MERGED_MODEL_URI
print(f"Repo name   : {REPO_NAME}")
print(f"Adapter URI : {MERGED_MODEL_URI}")


Repo name   : swift-json-vlm-container-finetuned
Adapter URI : gs://electric-bill-dataset-gcs/output/merged/v0-20260623-161620


## 4. Deploy to Vertex AI Endpoint

Uploads the container as a Vertex AI Model, then deploys it to an Endpoint with GPU serving. Any existing endpoint with the same `ENDPOINT_NAME` is torn down first so re-runs are idempotent.

In [43]:
import time
from google.api_core.exceptions import ServiceUnavailable

# Remove any previous endpoint with the same name (idempotent re-deploy)
existing = aiplatform.Endpoint.list(
    filter=f'display_name="{ENDPOINT_NAME}"',
    project=PROJECT_ID,
    location=REGION,
)
if existing:
    print(f"Removing old endpoint: {ENDPOINT_NAME}")
    existing[0].undeploy_all()
    existing[0].delete()

# --- Full env passed to the serving container ---
environment = {"ADAPTER_URI": MERGED_MODEL_URI, **SERVING_ENV_EXTRA}

# --- Register the container as a Vertex AI Model ---
model = aiplatform.Model.upload(
    display_name=DEPLOYED_MODEL_NAME,
    serving_container_image_uri=IMAGE_URI,
    serving_container_environment_variables=environment,
    serving_container_ports=[8080],
    serving_container_predict_route="/v1/chat/completions",
    serving_container_health_route="/health",
)
print(f"✅ Model uploaded: {model.resource_name}")

endpoint = aiplatform.Endpoint.create(display_name=ENDPOINT_NAME)
print(f"✅ Endpoint created: {endpoint.resource_name}")

Removing old endpoint: document-to-json
Deleting Endpoint : projects/73397202200/locations/asia-southeast1/endpoints/5917644148457865216


FailedPrecondition: 400 There are other operations running on the Endpoint "projects/73397202200/locations/asia-southeast1/endpoints/5917644148457865216". Operation(s) are: projects/73397202200/locations/asia-southeast1/operations/3431165397573828608.

In [ ]:
# Deploy with retry — Vertex AI GPU capacity is occasionally unavailable transiently (503),
# distinct from a genuine crash (400 FailedPrecondition, which does NOT retry — fix config instead).
deployed = False
for attempt in range(1, MAX_RETRIES + 1):
    try:
        print(f"\n🚀 Deploy attempt {attempt}/{MAX_RETRIES} — {MACHINE_TYPE} + {ACCELERATOR_TYPE} ...")
        model.deploy(
            endpoint=endpoint,
            machine_type=MACHINE_TYPE,
            accelerator_type=ACCELERATOR_TYPE,
            accelerator_count=ACCELERATOR_COUNT,
            min_replica_count=0,   
            max_replica_count=1,  
            traffic_percentage=100,
            sync=True,
            deploy_request_timeout=DEPLOY_TIMEOUT_SEC,
        )
        deployed = True
        print("✅ Deployed successfully!")
        break
    except ServiceUnavailable as e:
        print(f"⚠️  503 transient capacity error: {str(e)[:200]}")
        if attempt < MAX_RETRIES:
            print(f"   Retrying in {RETRY_WAIT_SEC}s...")
            time.sleep(RETRY_WAIT_SEC)
        else:
            print("❌ Exhausted retries. Try a different region or wait — capacity fluctuates.")
            raise
    # NOTE: a `FailedPrecondition: Model server exited unexpectedly` is NOT a capacity issue —
    # it means the container crashed on startup (commonly OOM on T4 16GB for this model size).
    # Check logs at the URL in the error message, or try MACHINE_TYPE="g2-standard-8" +
    # ACCELERATOR_TYPE="NVIDIA_L4" (24GB) in the Config cell above.


🚀 Deploy attempt 1/5 — n1-standard-8 + NVIDIA_TESLA_T4 ...
Deploying model to Endpoint : projects/73397202200/locations/asia-southeast1/endpoints/5917644148457865216
Deploy Endpoint model backing LRO: projects/73397202200/locations/asia-southeast1/endpoints/5917644148457865216/operations/3431165397573828608


In [ ]:
!gcloud alpha logging tail \
  'resource.type="aiplatform.googleapis.com/Endpoint" AND resource.labels.endpoint_id="319880917868871680"' \
  --project=first-orc-chien

You do not currently have this command group installed.  Using it 
requires the installation of components: [alpha]

ERROR: Cannot use bundled Python installation to update Google Cloud CLI in
non-interactive mode. Please run again in interactive mode.



If you really want to run in non-interactive mode, please run the
following command before re-running this one:



  FOR /F "delims=" %i in ( '""D:\Google\Cloud SDK\google-cloud-sdk\bin\gcloud.cmd"" components copy-bundled-python'
  ) DO (
    SET CLOUDSDK_PYTHON=%i
  )

(Substitute `%%i` for `%i` if in a .bat script.)
Installing component in a new window.

Please re-run this command when installation is complete.
    $ gcloud alpha logging tail 'resource.type=aiplatform.googleapis.com/Endpoint AND resource.labels.endpoint_id=319880917868871680' --project=first-orc-chien


In [ ]:
if deployed:
    print(f"\n✅ Endpoint live: {endpoint.resource_name}")
    with open("endpoint_config.json", "w") as f:
        json.dump({
            "endpoint_name":     ENDPOINT_NAME,
            "endpoint_resource": endpoint.resource_name,
            "model_name":        DEPLOYED_MODEL_NAME,
        }, f, indent=2)
    print("✅ Saved endpoint_config.json → used by 07_consume_model.ipynb")

## 5. (Optional) Debug Helpers

Not part of the normal deploy flow — use these only when a deploy fails and you need to inspect quota, GPU availability, or the built image.

In [ ]:
# Check current T4 quota for Vertex AI serving (NOT the same quota as Compute Engine GPUs)
print(f"""
Vertex AI serving GPU quota (separate from Compute Engine quota):
  https://console.cloud.google.com/iam-admin/quotas?project={PROJECT_ID}&service=aiplatform.googleapis.com

Search: "Custom model serving Nvidia T4 GPUs per region", filter Location = {REGION}
""")

In [ ]:
# List GPU types Vertex AI SDK supports, and what's actually available per zone via Compute Engine
from google.cloud import compute_v1
from google.cloud.aiplatform_v1.types import accelerator_type

def list_vertex_ai_supported_gpus():
    print("AcceleratorType values supported by the Vertex AI SDK:")
    for gpu in accelerator_type.AcceleratorType:
        if gpu.name != "ACCELERATOR_TYPE_UNSPECIFIED":
            print(f"  {gpu.name}")

def list_gpus_available_in_zone(project_id: str, zone: str):
    print(f"\nGPUs available in zone {zone} (Compute Engine API):")
    client = compute_v1.AcceleratorTypesClient()
    try:
        for accel in client.list(compute_v1.ListAcceleratorTypesRequest(project=project_id, zone=zone)):
            print(f"  - {accel.name}: {accel.description}")
    except Exception as e:
        print(f"  Error querying zone {zone}: {e}")

# Uncomment to run:
# list_vertex_ai_supported_gpus()
# for zone in [f"{REGION}-a", f"{REGION}-b", f"{REGION}-c"]:
#     list_gpus_available_in_zone(PROJECT_ID, zone)

In [ ]:
# Inspect the exact container spec Vertex AI registered for a given Model resource
# !gcloud ai models describe <MODEL_ID> --region={REGION} --format="yaml(containerSpec)"

In [ ]:
# Inspect files baked into the built image (sanity check entrypoint script is present)
# !docker run --rm --entrypoint sh {IMAGE_URI} -c "ls -la /home/vllmuser/app/"

In [ ]:
# Reload endpoint from disk in a fresh session and check which model(s) are currently deployed to it
with open("endpoint_config.json") as f:
    cfg = json.load(f)

endpoint = aiplatform.Endpoint(cfg["endpoint_resource"])
print(endpoint.gca_resource.deployed_models)

## Next Steps

Call the deployed endpoint for inference — see [07_consume_model.ipynb](./07_consume_model.ipynb).